In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [2]:
train = pd.read_csv("../data/raw/train.csv", dtype={"StateHoliday": str})
store = pd.read_csv("../data/raw/store.csv")

print("train shape:", train.shape)
print("store shape:", store.shape)

train shape: (1017209, 9)
store shape: (1115, 10)


In [8]:
print("---- train.csv missing values ----")
print(train.isnull().sum())

print("\n---- store.csv missing values ----")
print(store.isnull().sum())

---- train.csv missing values ----
Store            0
DayOfWeek        0
Date             0
Sales            0
Customers        0
Open             0
Promo            0
StateHoliday     0
SchoolHoliday    0
dtype: int64

---- store.csv missing values ----
Store                          0
StoreType                      0
Assortment                     0
CompetitionDistance            3
CompetitionOpenSinceMonth    354
CompetitionOpenSinceYear     354
Promo2                         0
Promo2SinceWeek              544
Promo2SinceYear              544
PromoInterval                544
dtype: int64


In [10]:
print("---- store.csv missing % ----")
missing_pct = (store.isnull().sum() / len(store) * 100).round(2)
print(missing_pct[missing_pct > 0])

---- store.csv missing % ----
CompetitionDistance           0.27
CompetitionOpenSinceMonth    31.75
CompetitionOpenSinceYear     31.75
Promo2SinceWeek              48.79
Promo2SinceYear              48.79
PromoInterval                48.79
dtype: float64


In [11]:
print(store['CompetitionDistance'].describe())
print('\nMissing:', store['CompetitionDistance'].isnull().sum())

count     1112.000000
mean      5404.901079
std       7663.174720
min         20.000000
25%        717.500000
50%       2325.000000
75%       6882.500000
max      75860.000000
Name: CompetitionDistance, dtype: float64

Missing: 3


In [12]:
print(store['CompetitionOpenSinceMonth'].value_counts(dropna=False).head(10))
print('\nMissing:', store['CompetitionOpenSinceMonth'].isnull().sum())

CompetitionOpenSinceMonth
NaN     354
9.0     125
4.0      94
11.0     92
3.0      70
7.0      67
12.0     64
10.0     61
6.0      50
5.0      44
Name: count, dtype: int64

Missing: 354


In [13]:
print(store["Promo2SinceWeek"].value_counts(dropna=False).head(5))
print("\nPromo2 value counts:")
print(store["Promo2"].value_counts())

Promo2SinceWeek
NaN     544
14.0     81
40.0     77
31.0     44
10.0     42
Name: count, dtype: int64

Promo2 value counts:
Promo2
1    571
0    544
Name: count, dtype: int64


In [15]:
store['CompetitionDistance'] = store['CompetitionDistance'].fillna(
    store['CompetitionDistance'].median()
)

print("Missing after fix:", store["CompetitionDistance"].isnull().sum())

Missing after fix: 0


In [16]:
store["CompetitionOpenSinceMonth"] = store["CompetitionOpenSinceMonth"].fillna(0)
store["CompetitionOpenSinceYear"] = store["CompetitionOpenSinceYear"].fillna(0)

print("CompetitionOpenSinceMonth missing:", store["CompetitionOpenSinceMonth"].isnull().sum())
print("CompetitionOpenSinceYear missing:", store["CompetitionOpenSinceYear"].isnull().sum())

CompetitionOpenSinceMonth missing: 0
CompetitionOpenSinceYear missing: 0


In [18]:
store['Promo2SinceWeek'] = store['Promo2SinceWeek'].fillna(0)
store['Promo2SinceYear'] = store['Promo2SinceYear'].fillna(0)
store['PromoInterval'] = store['PromoInterval'].fillna('None')

print("Promo2SinceWeek missing:", store["Promo2SinceWeek"].isnull().sum())
print("Promo2SinceYear missing:", store["Promo2SinceYear"].isnull().sum())
print("PromoInterval missing:", store["PromoInterval"].isnull().sum())

Promo2SinceWeek missing: 0
Promo2SinceYear missing: 0
PromoInterval missing: 0


In [19]:
print("=== store.csv after cleaning ===")
print(store.isnull().sum())
print("\nAll clean:", store.isnull().sum().sum() == 0)

=== store.csv after cleaning ===
Store                        0
StoreType                    0
Assortment                   0
CompetitionDistance          0
CompetitionOpenSinceMonth    0
CompetitionOpenSinceYear     0
Promo2                       0
Promo2SinceWeek              0
Promo2SinceYear              0
PromoInterval                0
dtype: int64

All clean: True


In [21]:
train['Date'] = pd.to_datetime(train['Date'])
train['StateHoliday'] = train['StateHoliday'].astype(str)

print(train.dtypes)

Store                     int64
DayOfWeek                 int64
Date             datetime64[ns]
Sales                     int64
Customers                 int64
Open                      int64
Promo                     int64
StateHoliday             object
SchoolHoliday             int64
dtype: object


In [22]:
open_zero_sales = train[(train['Open'] == 1) & (train['Sales'] == 0)]

print("Open=1 but Sales=0 rows:", len(open_zero_sales))
print("As % of open days:", round(len(open_zero_sales) / len(train[train["Open"]==1]) * 100, 2), "%")
print("\nStore samples:")
print(open_zero_sales["Store"].value_counts().head(10))

Open=1 but Sales=0 rows: 54
As % of open days: 0.01 %

Store samples:
Store
28      3
835     2
1039    2
665     2
623     2
1017    2
1100    2
25      2
102     2
364     2
Name: count, dtype: int64


## Decision: Dropping Open=1, Sales=0 rows

54 rows (0.01% of open days) where stores were marked open but recorded zero sales.
No single store accounts for more than 3 occurrences, ruling out a systematic issue.
Likely data entry errors. Decision: drop these rows before modelling.

In [23]:
# Filter: keep open days with actual sales only
train_clean = train[(train['Open'] == 1) & (train['Sales'] > 0)].copy()

# Merge with cleaned store data
merged_clean = train_clean.merge(store, on='Store', how='left')

print("Final shape:", merged_clean.shape)
print("Missing values:", merged_clean.isnull().sum().sum())


Final shape: (844338, 18)
Missing values: 0


In [31]:
print('--- Final merged_clean overview ----')
print(f'Rows:    {merged_clean.shape[0]:,}')
print(f'Columns: {merged_clean.shape[1]}')
print(f'\nDate range: {merged_clean['Date'].min()} to {merged_clean['Date'].max()}')
print(f'Unique stores: {merged_clean['Store'].nunique()}')
print(f'Missing values: {merged_clean.isnull().sum().sum()}')
print(f'\nColumns dtypes:\n{merged_clean.dtypes}')



--- Final merged_clean overview ----
Rows:    844,338
Columns: 18

Date range: 2013-01-01 00:00:00 to 2015-07-31 00:00:00
Unique stores: 1115
Missing values: 0

Columns dtypes:
Store                                 int64
DayOfWeek                             int64
Date                         datetime64[ns]
Sales                                 int64
Customers                             int64
Open                                  int64
Promo                                 int64
StateHoliday                         object
SchoolHoliday                         int64
StoreType                            object
Assortment                           object
CompetitionDistance                 float64
CompetitionOpenSinceMonth           float64
CompetitionOpenSinceYear            float64
Promo2                                int64
Promo2SinceWeek                     float64
Promo2SinceYear                     float64
PromoInterval                        object
dtype: object


In [32]:
os.makedirs('../data/processed', exist_ok=True)

merged_clean.to_csv('../data/processed/train_clean.csv', index=False)

print('Saved to ../data/processed/train_clean.csv')
print(f'File size: {os.path.getsize('../data/processed/train_clean.csv')}')

Saved to ../data/processed/train_clean.csv
File size: 67454116


## Cleaning Summary

### train.csv
- No missing values found
- Date converted from object to datetime64
- StateHoliday confirmed as string type
- 54 rows dropped (Open=1, Sales=0) — 0.01% of open days, likely data entry errors
- 172,817 closed-store rows excluded

### store.csv
- CompetitionDistance: 3 missing → filled with median (2,325m)
- CompetitionOpenSinceMonth/Year: 354 missing → filled with 0 (no competitor data)
- Promo2SinceWeek/Year: 544 missing → filled with 0 (stores not in Promo2 scheme)
- PromoInterval: 544 missing → filled with "None" (stores not in Promo2 scheme)

### Merged output
- Shape: (844,338, 18)
- Missing values: 0
- Saved to: data/processed/train_clean.csv